# Module 02 — Supervised Learning
## 02-01: Linear Regression

**Objective:** Derive ordinary least squares from maximum likelihood estimation, then implement both the closed-form normal equation and iterative gradient descent from scratch.

**Prerequisites:** 01-01 (NumPy & Linear Algebra), 01-09 (Optimization & Gradient Descent)

## Part 0 — Setup & Prerequisites

This notebook covers **linear regression** — the simplest and most foundational supervised learning model. We will derive the ordinary least squares (OLS) objective directly from maximum likelihood estimation, showing *why* minimizing squared error is the statistically principled choice under Gaussian noise. We then implement three solvers from scratch — the **normal equation** (exact, closed-form), **batch gradient descent** (iterative, full dataset), and **mini-batch stochastic gradient descent** (iterative, scalable) — apply all three to the California Housing dataset, and wrap up with residual diagnostics, a polynomial overfitting demonstration, a ridge regularization preview, and learning curve analysis.

Linear regression is the conceptual anchor for everything that follows — logistic regression (2-02), generalized linear models (2-06), and neural network loss minimization (Module 5) all build directly on the ideas introduced here.

**Prerequisites:** 01-01 (NumPy & Linear Algebra), 01-09 (Optimization & Gradient Descent)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch  # Used for gradient-descent demonstration (shows PyTorch is available)

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy:   {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU:  {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
import random

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"All random seeds set to SEED={SEED}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# Gradient descent — toy example (un-scaled features, needs small lr)
LEARNING_RATE_TOY: float = 0.01

# Gradient descent — California Housing (standardized features, can use larger lr)
LEARNING_RATE: float = 0.1
NUM_EPOCHS: int = 2000
TOLERANCE: float = 1e-7              # Early-stop when |Δloss| < TOLERANCE

# Mini-batch SGD
BATCH_SIZE: int = 256
SGD_LEARNING_RATE: float = 0.05
SGD_EPOCHS: int = 50

# Data
TEST_SIZE: float = 0.2

# Polynomial overfitting study
MAX_POLY_DEGREE: int = 6

# Paths
DATA_DIR: str = "../../data"

### Data Loading & EDA

We use the **California Housing** dataset (20,640 census block groups), which reports median house values and eight neighborhood features. It is a canonical regression benchmark because the target is continuous, the features are heterogeneous in scale, and the relationship between income and value is notably non-linear — making it a good stress test for linear models.

In [ ]:
# Load California Housing
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print("California Housing Dataset")
print(f"  Rows x Cols : {df.shape}")
print(f"  Features    : {list(housing.feature_names)}")
print(f"  Target      : MedHouseVal (median house value in $100,000s)")
print()
print("Target statistics:")
print(f"  Min  : {df['MedHouseVal'].min():.3f}  (${df['MedHouseVal'].min()*100:.0f}k)")
print(f"  Max  : {df['MedHouseVal'].max():.3f}  (${df['MedHouseVal'].max()*100:.0f}k)")
print(f"  Mean : {df['MedHouseVal'].mean():.3f}  (${df['MedHouseVal'].mean()*100:.0f}k)")
print(f"  Std  : {df['MedHouseVal'].std():.3f}")
print()
print(f"Missing values: {df.isnull().sum().sum()}")
df.describe()

In [ ]:
# ── EDA: Target Distribution & Feature Correlations ──────────────────────────
correlations = df.corr()["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: target distribution
axes[0].hist(df["MedHouseVal"], bins=60, color="steelblue", edgecolor="white")
axes[0].axvline(df["MedHouseVal"].mean(), color="red", linestyle="--",
                label=f"Mean = {df['MedHouseVal'].mean():.2f}")
axes[0].set_xlabel("Median House Value ($100k)")
axes[0].set_ylabel("Count")
axes[0].set_title("Target Distribution")
axes[0].legend()

# Right: Pearson correlations with target
bar_colors = ["#2ecc71" if c > 0 else "#e74c3c" for c in correlations]
axes[1].barh(correlations.index, correlations.values, color=bar_colors)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Pearson Correlation with MedHouseVal")
axes[1].set_title("Feature Correlations with Target")
for i, (val, name) in enumerate(zip(correlations.values, correlations.index)):
    offset = 0.01 * np.sign(val)
    axes[1].text(val + offset, i, f"{val:.3f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print(f"Strongest positive predictor : {correlations.idxmax()} (r = {correlations.max():.3f})")
print(f"Strongest negative predictor : {correlations.idxmin()} (r = {correlations.min():.3f})")

In [ ]:
# ── EDA: Top-4 Feature Scatter Plots ─────────────────────────────────────────
top_features = correlations.abs().nlargest(4).index.tolist()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, feature in zip(axes, top_features):
    ax.scatter(df[feature], df["MedHouseVal"], alpha=0.04, s=4, color="steelblue")
    ax.set_xlabel(feature)
    ax.set_ylabel("House Value ($100k)")
    ax.set_title(feature)

plt.suptitle("Top-4 Features by |Correlation| with Target", fontsize=13)
plt.tight_layout()
plt.show()

print("Observation: MedInc shows a clear positive trend but with saturation at high values.")
print("Latitude/Longitude reflect geographic clustering — coastal areas command price premiums.")

---
## Part 1 — Linear Regression from Scratch

### 1.1 Deriving OLS from Maximum Likelihood

We assume a **linear data-generating process** with additive Gaussian noise:

$$y_i = \mathbf{w}^\top \mathbf{x}_i + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, \sigma^2)$$

where $\mathbf{x}_i \in \mathbb{R}^p$ is the feature vector (with a prepended $1$ for the intercept), $\mathbf{w} \in \mathbb{R}^{p+1}$ is the weight vector, and $\sigma^2$ is the noise variance.

Under this model, the **conditional likelihood** of a single observation is:

$$p(y_i \mid \mathbf{x}_i, \mathbf{w}) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(y_i - \mathbf{w}^\top \mathbf{x}_i)^2}{2\sigma^2}\right)$$

Assuming i.i.d. samples, the **log-likelihood** over all $n$ observations is:

$$\ell(\mathbf{w}) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2} \sum_{i=1}^n (y_i - \mathbf{w}^\top \mathbf{x}_i)^2$$

Maximizing $\ell(\mathbf{w})$ with respect to $\mathbf{w}$ is **equivalent to minimizing the mean squared error**:

$$\mathcal{L}(\mathbf{w}) = \frac{1}{n} \|\mathbf{y} - \mathbf{X}\mathbf{w}\|^2$$

This is the **ordinary least squares (OLS)** objective. Its probabilistic justification tells us: *squaring the residuals is not an arbitrary choice — it follows from the Gaussian noise assumption.*

---

### 1.2 The Normal Equation (Closed-Form Solution)

Setting the gradient of $\mathcal{L}$ to zero:

$$\nabla_{\mathbf{w}} \mathcal{L} = -\frac{2}{n} \mathbf{X}^\top (\mathbf{y} - \mathbf{X}\mathbf{w}) = \mathbf{0}$$

Solving for $\mathbf{w}$ gives the **normal equation**:

$$\hat{\mathbf{w}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

In practice, we solve the linear system $(\mathbf{X}^\top\mathbf{X})\mathbf{w} = \mathbf{X}^\top\mathbf{y}$ via `np.linalg.solve` — this is numerically more stable than explicitly computing the inverse.

In [ ]:
# ── Normal Equation — Step by Step on Toy Data ───────────────────────────────
# True relationship: y = 2x + 1 + Gaussian noise
np.random.seed(SEED)
n_toy: int = 60
x_toy: np.ndarray = np.linspace(0, 5, n_toy)
y_toy: np.ndarray = 2.0 * x_toy + 1.0 + np.random.randn(n_toy) * 0.6

# Step 1: Build the design matrix — prepend a column of ones for the intercept
#         X shape: (n_samples, 2)  — column 0 = intercept (all 1s), column 1 = feature
X_toy: np.ndarray = np.column_stack([np.ones(n_toy), x_toy])
print(f"Design matrix X shape: {X_toy.shape}")

# Step 2: Compute the Gram matrix X^T X and cross-product X^T y
XtX: np.ndarray = X_toy.T @ X_toy   # shape: (2, 2)
Xty: np.ndarray = X_toy.T @ y_toy   # shape: (2,)
print(f"X^T X:\n{XtX}")
print(f"X^T y: {Xty}")

# Step 3: Solve (X^T X) w = X^T y for w — avoids explicit matrix inversion
w_normal: np.ndarray = np.linalg.solve(XtX, Xty)
print(f"\nNormal equation solution:")
print(f"  w0 (intercept) = {w_normal[0]:.4f}   true value = 1.0")
print(f"  w1 (slope)     = {w_normal[1]:.4f}   true value = 2.0")

# Visualize the fit
x_line: np.ndarray = np.array([0.0, 5.0])
X_line: np.ndarray = np.column_stack([np.ones(2), x_line])
y_line_normal: np.ndarray = X_line @ w_normal

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_toy, y_toy, alpha=0.7, color="steelblue", label="Noisy observations")
ax.plot(x_line, y_line_normal, "r-", linewidth=2.5,
        label=f"Normal Eq: y = {w_normal[1]:.3f}x + {w_normal[0]:.3f}")
ax.plot(x_line, 2 * x_line + 1, "g--", linewidth=1.5, alpha=0.8, label="True: y = 2x + 1")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Normal Equation: Closed-Form OLS Fit")
ax.legend()
plt.tight_layout()
plt.show()

### 1.3 Gradient Descent

When $p$ (number of features) is very large, forming and factoring $(\mathbf{X}^\top\mathbf{X})$ becomes expensive. **Gradient descent** instead iteratively updates weights by stepping in the direction of steepest descent:

$$\nabla_{\mathbf{w}} \mathcal{L}(\mathbf{w}) = -\frac{2}{n} \mathbf{X}^\top (\mathbf{y} - \mathbf{X}\mathbf{w})$$

$$\mathbf{w}^{(t+1)} \leftarrow \mathbf{w}^{(t)} - \alpha \cdot \nabla_{\mathbf{w}} \mathcal{L}(\mathbf{w}^{(t)})$$

Because the MSE objective is strictly convex, gradient descent converges to the same global minimum as the normal equation — it just takes many steps.

**Feature standardization is essential for gradient descent convergence.** Without it, the loss landscape is very elongated (high curvature in some directions, low in others) and gradient descent oscillates or converges slowly. After standardization, the landscape is more spherical and gradient descent converges quickly with a larger learning rate.

In [ ]:
def compute_mse_and_gradient(
    X: np.ndarray,
    y: np.ndarray,
    weights: np.ndarray,
) -> tuple[float, np.ndarray]:
    """Compute MSE loss and its gradient with respect to the weight vector.

    Loss:     L(w) = (1/n) * ||y - Xw||^2
    Gradient: grad_w L = -(2/n) * X^T (y - Xw)

    Args:
        X: Design matrix of shape (n_samples, n_features). Must include bias column.
        y: Target vector of shape (n_samples,).
        weights: Current weight vector of shape (n_features,).

    Returns:
        Tuple of (mse_loss, gradient_vector).
    """
    n_samples: int = X.shape[0]
    residuals: np.ndarray = y - X @ weights
    mse_loss: float = float(np.mean(residuals ** 2))
    gradient: np.ndarray = -(2.0 / n_samples) * (X.T @ residuals)
    return mse_loss, gradient


def check_gradient_numerically(
    X: np.ndarray,
    y: np.ndarray,
    weights: np.ndarray,
    epsilon: float = 1e-5,
) -> np.ndarray:
    """Numerically approximate the gradient via finite differences (for verification).

    Uses the central difference formula: df/dw_i ~ [f(w+eps) - f(w-eps)] / (2*eps).

    Args:
        X: Design matrix of shape (n_samples, n_features).
        y: Target vector of shape (n_samples,).
        weights: Weight vector at which to evaluate the gradient.
        epsilon: Finite difference step size.

    Returns:
        Numerically approximated gradient of shape (n_features,).
    """
    numerical_grad: np.ndarray = np.zeros_like(weights)
    for i in range(len(weights)):
        w_plus = weights.copy()
        w_minus = weights.copy()
        w_plus[i] += epsilon
        w_minus[i] -= epsilon
        loss_plus, _ = compute_mse_and_gradient(X, y, w_plus)
        loss_minus, _ = compute_mse_and_gradient(X, y, w_minus)
        numerical_grad[i] = (loss_plus - loss_minus) / (2.0 * epsilon)
    return numerical_grad


# Gradient check: compare analytical vs numerical gradient
w_test: np.ndarray = np.array([0.5, 1.2])  # arbitrary test point
analytical_grad: np.ndarray = compute_mse_and_gradient(X_toy, y_toy, w_test)[1]
numerical_grad: np.ndarray = check_gradient_numerically(X_toy, y_toy, w_test)
print("Gradient check (analytical vs numerical finite differences):")
print(f"  Analytical : {analytical_grad}")
print(f"  Numerical  : {numerical_grad}")
print(f"  Max error  : {np.max(np.abs(analytical_grad - numerical_grad)):.2e}  (should be < 1e-5)")

In [ ]:
# ── Batch Gradient Descent on Toy Data ───────────────────────────────────────
weights_gd: np.ndarray = np.zeros(X_toy.shape[1])  # initialize at origin
gd_toy_loss_history: list[float] = []
gd_toy_weight_history: list[np.ndarray] = []  # track path through weight space

for epoch in range(NUM_EPOCHS):
    loss, gradient = compute_mse_and_gradient(X_toy, y_toy, weights_gd)
    gd_toy_loss_history.append(loss)
    gd_toy_weight_history.append(weights_gd.copy())
    weights_gd = weights_gd - LEARNING_RATE_TOY * gradient
    # Early stopping: stop when loss improvement drops below tolerance
    if epoch > 0 and abs(gd_toy_loss_history[-2] - loss) < TOLERANCE:
        print(f"Converged at epoch {epoch + 1}")
        break

print(f"\nGradient Descent solution (lr={LEARNING_RATE_TOY}, {len(gd_toy_loss_history)} steps):")
print(f"  w0 (intercept) = {weights_gd[0]:.4f}   true value = 1.0")
print(f"  w1 (slope)     = {weights_gd[1]:.4f}   true value = 2.0")
print(f"  Final MSE      = {gd_toy_loss_history[-1]:.6f}")

print(f"\nNormal Equation solution for comparison:")
print(f"  w0 (intercept) = {w_normal[0]:.4f}")
print(f"  w1 (slope)     = {w_normal[1]:.4f}")
print(f"  Max weight diff: {np.max(np.abs(w_normal - weights_gd)):.2e}")

In [ ]:
# ── Visualize GD Convergence and Both Fits ────────────────────────────────────
y_line_gd: np.ndarray = X_line @ weights_gd

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: convergence curve on log scale
axes[0].plot(gd_toy_loss_history, color="darkorange", linewidth=1.5)
axes[0].set_xlabel("Gradient Descent Iteration")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title(f"GD Convergence (lr={LEARNING_RATE_TOY})")
axes[0].set_yscale("log")
axes[0].grid(True, alpha=0.3)
axes[0].axhline(gd_toy_loss_history[-1], color="gray", linestyle=":",
                label=f"Final MSE = {gd_toy_loss_history[-1]:.4f}")
axes[0].legend()

# Right: overlay both fitted lines on the scatter plot
axes[1].scatter(x_toy, y_toy, alpha=0.7, color="steelblue", s=25, label="Data")
axes[1].plot(x_line, y_line_normal, "r-", linewidth=2.5, label="Normal Equation (exact)")
axes[1].plot(x_line, y_line_gd, "g--", linewidth=2.5,
             label=f"Gradient Descent ({len(gd_toy_loss_history)} steps)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Normal Equation vs Gradient Descent")
axes[1].legend()

plt.tight_layout()
plt.show()
print("Both methods converge to the same solution.")

In [ ]:
# ── Weight-Space Trajectory ───────────────────────────────────────────────────
# Visualize how gradient descent walks through (w0, w1) space toward the OLS minimum.
# The contours show the MSE loss surface; the path shows the GD trajectory.

# Build a fine grid around the optimum for the contour plot
w0_opt, w1_opt = w_normal[0], w_normal[1]
w0_range: np.ndarray = np.linspace(w0_opt - 2.5, w0_opt + 2.5, 120)
w1_range: np.ndarray = np.linspace(w1_opt - 2.5, w1_opt + 2.5, 120)
W0, W1 = np.meshgrid(w0_range, w1_range)

# Compute MSE at each grid point
loss_grid: np.ndarray = np.zeros_like(W0)
for i in range(W0.shape[0]):
    for j in range(W0.shape[1]):
        w_grid = np.array([W0[i, j], W1[i, j]])
        loss_grid[i, j], _ = compute_mse_and_gradient(X_toy, y_toy, w_grid)

# Extract GD path from recorded weight history
weight_path: np.ndarray = np.array(gd_toy_weight_history)

# Subsample path for cleaner arrow visualization (every N steps)
arrow_step: int = max(1, len(weight_path) // 20)
path_w0: np.ndarray = weight_path[:, 0]
path_w1: np.ndarray = weight_path[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: filled contour of the loss surface with GD path
contour = axes[0].contourf(W0, W1, loss_grid, levels=40, cmap="viridis")
plt.colorbar(contour, ax=axes[0], label="MSE Loss")
axes[0].plot(path_w0, path_w1, "w-", linewidth=1.5, alpha=0.8, label="GD path")
axes[0].plot(path_w0[0], path_w1[0], "ws", markersize=10, label="Start (0, 0)")
axes[0].plot(w0_opt, w1_opt, "r*", markersize=14, label=f"Optimum ({w0_opt:.2f}, {w1_opt:.2f})")
# Draw arrows along the path to show direction
for k in range(0, len(path_w0) - arrow_step, arrow_step):
    axes[0].annotate(
        "",
        xy=(path_w0[k + arrow_step], path_w1[k + arrow_step]),
        xytext=(path_w0[k], path_w1[k]),
        arrowprops=dict(arrowstyle="->", color="white", lw=1.5),
    )
axes[0].set_xlabel("w0 (intercept)")
axes[0].set_ylabel("w1 (slope)")
axes[0].set_title("MSE Loss Surface with GD Trajectory")
axes[0].legend(fontsize=8)

# Right: log-loss for each step colored by iteration index
sc = axes[1].scatter(
    range(len(gd_toy_loss_history)),
    np.log10(gd_toy_loss_history),
    c=range(len(gd_toy_loss_history)),
    cmap="plasma", s=8,
)
plt.colorbar(sc, ax=axes[1], label="Iteration")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("log10(MSE)")
axes[1].set_title("Loss Decrease per Iteration")

plt.tight_layout()
plt.show()
print("The GD path spirals inward toward the minimum — faster early on, slower near convergence.")

---
## Part 2 — Putting It All Together

We now wrap the normal equation and two gradient-based solvers into a single `LinearRegressionScratch` class with a scikit-learn-style API. We also implement **mini-batch stochastic gradient descent (SGD)**, which processes one random mini-batch of the training data per update rather than the full dataset. This is the conceptual bridge to neural network training in Module 5 — where SGD on mini-batches is the universal default.

| Solver | Complexity per step | Memory | Convergence |
|--------|--------------------|---------|--------------|
| Normal equation | $\mathcal{O}(np^2 + p^3)$ once | $\mathcal{O}(p^2)$ | Exact |
| Batch GD | $\mathcal{O}(np)$ per epoch | $\mathcal{O}(np)$ | Slow but smooth |
| Mini-batch SGD | $\mathcal{O}(Bp)$ per step | $\mathcal{O}(Bp)$ | Noisy but fast per epoch |

In [ ]:
class LinearRegressionScratch:
    """Linear regression from scratch — normal equation, batch GD, or mini-batch SGD.

    All three solvers minimize the mean squared error:
        L(w) = (1/n) * ||y - Xw||^2

    Attributes:
        method: Solver — 'normal', 'gradient_descent', or 'sgd'.
        learning_rate: Step size for gradient-based solvers.
        num_epochs: Maximum gradient-based iterations.
        batch_size: Mini-batch size (only used by 'sgd').
        tolerance: Early-stop threshold on consecutive loss improvement.
        weights_: Learned weight vector of shape (n_features+1,). Index 0 = intercept.
        loss_history_: Training MSE per epoch (empty for 'normal' method).
    """

    def __init__(
        self,
        method: str = "normal",
        learning_rate: float = 0.01,
        num_epochs: int = 2000,
        batch_size: int = 64,
        tolerance: float = 1e-7,
    ) -> None:
        """Initialize the linear regression model.

        Args:
            method: Solver — 'normal', 'gradient_descent', or 'sgd'.
            learning_rate: Step size (ignored for 'normal').
            num_epochs: Max iterations (ignored for 'normal').
            batch_size: Mini-batch size for 'sgd'.
            tolerance: Stop when |change in loss| < tolerance (ignored for 'normal').
        """
        if method not in ("normal", "gradient_descent", "sgd"):
            raise ValueError(f"method must be 'normal', 'gradient_descent', or 'sgd'.")
        self.method = method
        self.learning_rate = learning_rate
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.tolerance = tolerance
        self.weights_: np.ndarray | None = None
        self.loss_history_: list[float] = []

    def _add_bias(
        self,
        X: np.ndarray,
    ) -> np.ndarray:
        """Prepend a column of ones to X for the bias (intercept) term.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Design matrix of shape (n_samples, n_features + 1).
        """
        return np.column_stack([np.ones(X.shape[0]), X])

    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
    ) -> "LinearRegressionScratch":
        """Fit the model to training data using the configured solver.

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            y: Target vector of shape (n_samples,).

        Returns:
            self, enabling method chaining (e.g., model.fit(X, y).predict(X_test)).
        """
        X_design: np.ndarray = self._add_bias(X)
        n_samples, n_params = X_design.shape

        if self.method == "normal":
            # Solve (X^T X) w = X^T y  — never invert; use a linear system solver
            self.weights_ = np.linalg.solve(X_design.T @ X_design, X_design.T @ y)

        elif self.method == "gradient_descent":
            # Full-batch gradient descent: gradient uses ALL training samples each step
            self.weights_ = np.zeros(n_params)
            self.loss_history_ = []
            for epoch in range(self.num_epochs):
                loss, grad = compute_mse_and_gradient(X_design, y, self.weights_)
                self.loss_history_.append(loss)
                self.weights_ = self.weights_ - self.learning_rate * grad
                if epoch > 0 and abs(self.loss_history_[-2] - loss) < self.tolerance:
                    break

        else:  # sgd
            # Mini-batch SGD: each step uses a random subset of batch_size samples
            self.weights_ = np.zeros(n_params)
            self.loss_history_ = []
            rng = np.random.default_rng(seed=SEED)
            for epoch in range(self.num_epochs):
                # Shuffle indices and split into mini-batches
                indices: np.ndarray = rng.permutation(n_samples)
                epoch_loss_sum: float = 0.0
                n_batches: int = 0
                for start in range(0, n_samples, self.batch_size):
                    batch_idx = indices[start: start + self.batch_size]
                    X_batch: np.ndarray = X_design[batch_idx]
                    y_batch: np.ndarray = y[batch_idx]
                    loss_batch, grad_batch = compute_mse_and_gradient(
                        X_batch, y_batch, self.weights_
                    )
                    self.weights_ = self.weights_ - self.learning_rate * grad_batch
                    epoch_loss_sum += loss_batch
                    n_batches += 1
                # Record full-batch MSE at the end of each epoch for comparability
                epoch_loss, _ = compute_mse_and_gradient(X_design, y, self.weights_)
                self.loss_history_.append(epoch_loss)
                if epoch > 0 and abs(self.loss_history_[-2] - epoch_loss) < self.tolerance:
                    break

        return self

    def predict(
        self,
        X: np.ndarray,
    ) -> np.ndarray:
        """Predict target values for input data.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Predicted values of shape (n_samples,).

        Raises:
            RuntimeError: If called before fit().
        """
        if self.weights_ is None:
            raise RuntimeError("Model not fitted. Call fit() before predict().")
        return self._add_bias(X) @ self.weights_

    @property
    def intercept_(self) -> float:
        """Return the learned intercept (bias) term."""
        if self.weights_ is None:
            raise RuntimeError("Model not fitted.")
        return float(self.weights_[0])

    @property
    def coef_(self) -> np.ndarray:
        """Return the learned feature coefficients (excluding intercept)."""
        if self.weights_ is None:
            raise RuntimeError("Model not fitted.")
        return self.weights_[1:]

In [ ]:
# ── Sanity Check: Recover Known Ground Truth ──────────────────────────────────
np.random.seed(SEED)
n_sanity: int = 300
X_sanity: np.ndarray = np.random.randn(n_sanity, 3)
true_coefs: np.ndarray = np.array([1.5, -2.0, 0.8])
true_intercept: float = 0.5
y_sanity: np.ndarray = (
    X_sanity @ true_coefs + true_intercept + 0.1 * np.random.randn(n_sanity)
)

print(f"True intercept : {true_intercept}")
print(f"True coefs     : {true_coefs}")
print()
print(f"{'Method':<22} {'intercept':>10} {'coef[0]':>8} {'coef[1]':>8} {'coef[2]':>8} {'Train MSE':>12}")
print("-" * 75)

for method_name in ["normal", "gradient_descent", "sgd"]:
    model_sanity = LinearRegressionScratch(
        method=method_name,
        learning_rate=0.05,
        num_epochs=200,
        batch_size=32,
        tolerance=1e-9,
    )
    model_sanity.fit(X_sanity, y_sanity)
    train_mse = float(mean_squared_error(y_sanity, model_sanity.predict(X_sanity)))
    print(
        f"{method_name:<22} {model_sanity.intercept_:>10.4f} "
        f"{model_sanity.coef_[0]:>8.4f} {model_sanity.coef_[1]:>8.4f} "
        f"{model_sanity.coef_[2]:>8.4f} {train_mse:>12.6f}"
    )

print()
print("All three solvers recover the true weights — small differences are due to finite epochs.")

---
## Part 3 — Training on California Housing

We now apply `LinearRegressionScratch` to the full California Housing dataset. The workflow is:
1. Split into 80% train / 20% test using `train_test_split`
2. Standardize features using `StandardScaler` (fit on train only — **never peek at test statistics**)
3. Fit three models: normal equation, batch GD, and mini-batch SGD
4. Compare against sklearn's optimized solver
5. Measure the effect of training set size (learning curves)

In [ ]:
# ── Train/Test Split & Feature Standardization ───────────────────────────────
X_raw: np.ndarray = housing.data.values    # shape: (20640, 8)
y_raw: np.ndarray = housing.target.values  # shape: (20640,)

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=TEST_SIZE, random_state=SEED
)

# Fit scaler on training data only — prevents information leakage from test set
scaler = StandardScaler()
X_train_scaled: np.ndarray = scaler.fit_transform(X_train)
X_test_scaled: np.ndarray = scaler.transform(X_test)

print(f"Train : {X_train_scaled.shape[0]} samples ({1 - TEST_SIZE:.0%})")
print(f"Test  : {X_test_scaled.shape[0]} samples ({TEST_SIZE:.0%})")
print(f"Feats : {X_train_scaled.shape[1]}")
print(f"\nAfter StandardScaler — feature means  : {X_train_scaled.mean(axis=0).round(4)}")
print(f"After StandardScaler — feature stds   : {X_train_scaled.std(axis=0).round(4)}")

In [ ]:
# ── Fit All Three Solvers ─────────────────────────────────────────────────────
solvers: dict = {
    "normal": LinearRegressionScratch(method="normal"),
    "gradient_descent": LinearRegressionScratch(
        method="gradient_descent",
        learning_rate=LEARNING_RATE,
        num_epochs=NUM_EPOCHS,
        tolerance=TOLERANCE,
    ),
    "sgd": LinearRegressionScratch(
        method="sgd",
        learning_rate=SGD_LEARNING_RATE,
        num_epochs=SGD_EPOCHS,
        batch_size=BATCH_SIZE,
        tolerance=TOLERANCE,
    ),
}

fit_results: dict = {}
for solver_name, model in solvers.items():
    t_start = time.perf_counter()
    model.fit(X_train_scaled, y_train)
    elapsed = time.perf_counter() - t_start
    y_pred_test = model.predict(X_test_scaled)
    test_mse = float(mean_squared_error(y_test, y_pred_test))
    epochs_str = str(len(model.loss_history_)) if model.loss_history_ else "N/A"
    fit_results[solver_name] = {"model": model, "y_pred_test": y_pred_test,
                                "test_mse": test_mse, "time_ms": elapsed * 1000}
    print(f"[{solver_name:>20}]  time={elapsed*1000:6.1f}ms  "
          f"epochs={epochs_str:>5}  test_MSE={test_mse:.4f}")

# Also store predictions for later cells
y_pred_normal_test: np.ndarray = fit_results["normal"]["y_pred_test"]
y_pred_gd_test: np.ndarray = fit_results["gradient_descent"]["y_pred_test"]
y_pred_sgd_test: np.ndarray = fit_results["sgd"]["y_pred_test"]
model_normal_full: LinearRegressionScratch = fit_results["normal"]["model"]
model_gd_full: LinearRegressionScratch = fit_results["gradient_descent"]["model"]

In [ ]:
# ── Solver Convergence Comparison: Batch GD vs Mini-Batch SGD ────────────────
fig, ax = plt.subplots(figsize=(10, 4))

gd_model: LinearRegressionScratch = fit_results["gradient_descent"]["model"]
sgd_model: LinearRegressionScratch = fit_results["sgd"]["model"]

ax.plot(gd_model.loss_history_, color="darkorange", linewidth=1.8, label="Batch GD")
ax.plot(sgd_model.loss_history_, color="steelblue", linewidth=1.8,
        linestyle="--", label=f"Mini-batch SGD (batch={BATCH_SIZE})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Full-batch Training MSE")
ax.set_title("Batch GD vs Mini-batch SGD: Convergence on California Housing")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Batch GD      : {len(gd_model.loss_history_)} epochs, "
      f"final train MSE = {gd_model.loss_history_[-1]:.4f}")
print(f"Mini-batch SGD: {len(sgd_model.loss_history_)} epochs, "
      f"final train MSE = {sgd_model.loss_history_[-1]:.4f}")
print("\nMini-batch SGD converges in far fewer epochs — crucial at large scale.")

In [ ]:
# ── Learning Curves: Train/Test MSE vs Training Set Size ─────────────────────
# Shows whether the model is in the underfitting or overfitting regime.
# If train and test error are both high -> underfitting (high bias).
# If train error is much lower than test -> overfitting (high variance).

train_sizes: list[float] = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
lc_train_mses: list[float] = []
lc_test_mses: list[float] = []
lc_n_samples: list[int] = []

for fraction in train_sizes:
    n_use: int = max(10, int(fraction * len(X_train_scaled)))
    X_sub: np.ndarray = X_train_scaled[:n_use]
    y_sub: np.ndarray = y_train[:n_use]

    model_lc = LinearRegressionScratch(method="normal")
    model_lc.fit(X_sub, y_sub)

    train_mse_lc = float(mean_squared_error(y_sub, model_lc.predict(X_sub)))
    test_mse_lc = float(mean_squared_error(y_test, model_lc.predict(X_test_scaled)))
    lc_train_mses.append(train_mse_lc)
    lc_test_mses.append(test_mse_lc)
    lc_n_samples.append(n_use)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(lc_n_samples, lc_train_mses, "o-", color="steelblue", label="Train MSE")
ax.plot(lc_n_samples, lc_test_mses, "s--", color="darkorange", label="Test MSE")
ax.fill_between(lc_n_samples, lc_train_mses, lc_test_mses, alpha=0.15, color="gray",
                label="Generalization gap")
ax.set_xlabel("Training Set Size")
ax.set_ylabel("MSE")
ax.set_title("Learning Curves — Linear Regression on California Housing")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - Train MSE rises as more data is added (model can no longer memorize).")
print("  - Test MSE falls as training set grows (more data = better generalization).")
print("  - The two curves converge to a plateau — this is the irreducible bias of linear models.")
print(f"  - Gap at full training size: {lc_test_mses[-1] - lc_train_mses[-1]:.4f}")
print("  - A large persistent gap would indicate overfitting; here, both converge -> high bias.")

In [ ]:
# ── Feature Engineering: Degree-2 Polynomial on All 8 Features ───────────────
# Can we improve beyond R^2 ~ 0.60 by adding all pairwise interactions and squares?
# Degree-2 polynomial on 8 features yields 44 features total:
#   8 original + 8 squared terms + 28 pairwise cross-products.
# This captures non-linear effects (MedInc^2) and interactions (MedInc x Latitude).

poly_all = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly_all: np.ndarray = poly_all.fit_transform(X_train_scaled)
X_test_poly_all: np.ndarray = poly_all.transform(X_test_scaled)

# Standardize the expanded matrix — essential because squared features have large variance
scaler_poly_all = StandardScaler()
X_train_poly_sc: np.ndarray = scaler_poly_all.fit_transform(X_train_poly_all)
X_test_poly_sc: np.ndarray = scaler_poly_all.transform(X_test_poly_all)

n_original: int = X_train_scaled.shape[1]
n_expanded: int = X_train_poly_all.shape[1]
print(f"Original features   : {n_original}")
print(f"Degree-2 expansion  : {n_expanded} features")
print(f"  = {n_original} original + {n_original} squared + {n_original*(n_original-1)//2} pairwise")

# Fit normal equation — now solving a 44x44 Gram matrix
model_poly_all = LinearRegressionScratch(method="normal")
t_start = time.perf_counter()
model_poly_all.fit(X_train_poly_sc, y_train)
t_poly_all = time.perf_counter() - t_start

y_pred_poly_all: np.ndarray = model_poly_all.predict(X_test_poly_sc)

poly_mse: float = float(mean_squared_error(y_test, y_pred_poly_all))
poly_rmse: float = float(np.sqrt(poly_mse))
poly_r2: float = float(r2_score(y_test, y_pred_poly_all))
linear_mse: float = float(mean_squared_error(y_test, y_pred_normal_test))
linear_r2: float = float(r2_score(y_test, y_pred_normal_test))

feat_eng_df = pd.DataFrame({
    "Model": [f"Linear ({n_original} features)", f"Degree-2 Poly ({n_expanded} features)"],
    "MSE": [round(linear_mse, 4), round(poly_mse, 4)],
    "RMSE": [round(float(np.sqrt(linear_mse)), 4), round(poly_rmse, 4)],
    "R2": [round(linear_r2, 4), round(poly_r2, 4)],
})
print(f"\nPerformance comparison (poly fit time: {t_poly_all*1000:.1f}ms):")
print(feat_eng_df.to_string(index=False))

r2_gain: float = poly_r2 - linear_r2
mse_reduction: float = (linear_mse - poly_mse) / linear_mse
print(f"\nR2 improvement : +{r2_gain:.4f}")
print(f"MSE reduction  : {mse_reduction:.1%}")

# Identify the top-10 most influential polynomial features
poly_feature_names: list[str] = poly_all.get_feature_names_out(input_features=list(housing.feature_names))
coef_magnitudes: np.ndarray = np.abs(model_poly_all.coef_)
top_indices: np.ndarray = np.argsort(coef_magnitudes)[::-1][:10]
print(f"\nTop-10 most influential polynomial features (by |coefficient|):")
for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank:2d}. {poly_feature_names[idx]:<30} coef = {model_poly_all.coef_[idx]:>8.4f}")

print()
print("Degree-2 polynomial on all 8 features substantially reduces error by capturing:")
print("  - Non-linear effects: e.g., MedInc^2 (diminishing price returns at high income)")
print("  - Interaction effects: e.g., MedInc x Latitude (income premium varies by region)")
print("The tradeoff: 44 params vs 8, more risk of overfitting on smaller datasets,")
print("and the Gram matrix is now 44x44 instead of 8x8.")
print("Ridge regularization (shown next) mitigates overfitting in this expanded feature space.")

### Polynomial Features & Overfitting

Linear regression can only fit **hyperplane** decision surfaces. When the true relationship is curved, we can enrich the feature space with **polynomial features** — e.g., adding $x^2$, $x^3$, cross-products — and run OLS on the expanded matrix.

However, high-degree polynomials have so many parameters that they can interpolate the training data almost perfectly while **generalizing poorly** to new data. This is the classic **overfitting** signature: train error falls monotonically with degree, while test error has a U-shape. We illustrate this with the **MedInc** feature (strongest single predictor) for easy 2-D visualization.

In [ ]:
# ── Polynomial Degree vs Train/Test MSE ──────────────────────────────────────
# Use only MedInc (index 0, un-scaled) for 2-D visualization
X_medinc_train: np.ndarray = X_train[:, 0:1]
X_medinc_test: np.ndarray = X_test[:, 0:1]

degrees: list[int] = list(range(1, MAX_POLY_DEGREE + 1))
poly_train_mses: list[float] = []
poly_test_mses: list[float] = []
saved_fits: dict[int, tuple] = {}  # store for visualization

for degree in degrees:
    poly_transformer = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly_train: np.ndarray = poly_transformer.fit_transform(X_medinc_train)
    X_poly_test: np.ndarray = poly_transformer.transform(X_medinc_test)

    scaler_poly = StandardScaler()
    X_poly_train_sc: np.ndarray = scaler_poly.fit_transform(X_poly_train)
    X_poly_test_sc: np.ndarray = scaler_poly.transform(X_poly_test)

    model_poly = LinearRegressionScratch(method="normal")
    model_poly.fit(X_poly_train_sc, y_train)

    train_mse_p = float(mean_squared_error(y_train, model_poly.predict(X_poly_train_sc)))
    test_mse_p = float(mean_squared_error(y_test, model_poly.predict(X_poly_test_sc)))
    poly_train_mses.append(train_mse_p)
    poly_test_mses.append(test_mse_p)

    if degree in (1, 2, MAX_POLY_DEGREE):
        saved_fits[degree] = (model_poly, poly_transformer, scaler_poly)

    print(f"  Degree {degree} | params={poly_transformer.n_output_features_:>3} | "
          f"Train MSE={train_mse_p:.4f} | Test MSE={test_mse_p:.4f}")

best_degree: int = degrees[int(np.argmin(poly_test_mses))]
print(f"\nBest degree by test MSE: {best_degree}")

In [ ]:
# ── Visualize the Bias-Variance Tradeoff ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MSE curves per degree
axes[0].plot(degrees, poly_train_mses, "o-", color="steelblue", label="Train MSE")
axes[0].plot(degrees, poly_test_mses, "s--", color="darkorange", label="Test MSE")
axes[0].axvline(best_degree, color="gray", linestyle=":", label=f"Best degree={best_degree}")
axes[0].set_xlabel("Polynomial Degree")
axes[0].set_ylabel("MSE")
axes[0].set_title("Polynomial Degree vs MSE — Bias-Variance Tradeoff")
axes[0].legend()
axes[0].set_xticks(degrees)

# Right: fitted curves for degree 1, 2, and MAX
x_vis: np.ndarray = np.linspace(X_medinc_train.min(), X_medinc_train.max(), 300).reshape(-1, 1)
plot_colors: dict = {1: "steelblue", 2: "#2ecc71", MAX_POLY_DEGREE: "#e74c3c"}

for degree_d, (model_d, poly_d, scaler_d) in saved_fits.items():
    X_vis_poly: np.ndarray = scaler_d.transform(poly_d.transform(x_vis))
    y_vis: np.ndarray = model_d.predict(X_vis_poly)
    axes[1].plot(x_vis.flatten(), y_vis, linewidth=2.5,
                 color=plot_colors[degree_d], label=f"Degree {degree_d}")

axes[1].scatter(X_medinc_train.flatten(), y_train,
                alpha=0.05, s=6, color="gray", label="Training data")
axes[1].set_ylim(-0.5, 6.0)
axes[1].set_xlabel("Median Income (MedInc)")
axes[1].set_ylabel("House Value ($100k)")
axes[1].set_title(f"Polynomial Fits: Degree 1, 2, {MAX_POLY_DEGREE}")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Degree 1: underfits (too rigid). Degree {MAX_POLY_DEGREE}: overfits (follows noise).")
print(f"Degree {best_degree}: best generalization — lowest test MSE.")

### Library Comparison

A critical validation step: our from-scratch normal equation should produce **numerically identical** coefficients to `sklearn.linear_model.LinearRegression`, which internally uses LAPACK's least-squares routines. Any significant discrepancy would indicate a bug.

In [ ]:
# ── sklearn LinearRegression Comparison ──────────────────────────────────────
sklearn_lr = LinearRegression(fit_intercept=True)
sklearn_lr.fit(X_train_scaled, y_train)
y_pred_sklearn: np.ndarray = sklearn_lr.predict(X_test_scaled)

print("Coefficient comparison (from-scratch Normal Eq vs sklearn LinearRegression):")
print(f"\n{'Feature':<14} {'From Scratch':>14} {'sklearn':>12} {'|Diff|':>12}")
print("-" * 57)
for fname, w_scratch, w_sk in zip(
    housing.feature_names, model_normal_full.coef_, sklearn_lr.coef_
):
    diff = abs(w_scratch - w_sk)
    print(f"{fname:<14} {w_scratch:>14.6f} {w_sk:>12.6f} {diff:>12.2e}")

int_diff = abs(model_normal_full.intercept_ - sklearn_lr.intercept_)
print("-" * 57)
print(f"{'Intercept':<14} {model_normal_full.intercept_:>14.6f} "
      f"{sklearn_lr.intercept_:>12.6f} {int_diff:>12.2e}")

test_mse_sklearn = float(mean_squared_error(y_test, y_pred_sklearn))
test_mse_scratch = float(mean_squared_error(y_test, y_pred_normal_test))
coefs_match = bool(np.allclose(model_normal_full.coef_, sklearn_lr.coef_, atol=1e-4))
print(f"\nTest MSE (from-scratch) : {test_mse_scratch:.6f}")
print(f"Test MSE (sklearn)      : {test_mse_sklearn:.6f}")
print(f"Coefficients match      : {coefs_match}")

### Ridge Regularization — Preview

The polynomial overfitting experiment showed that unconstrained OLS can learn very large weights that track noise. **Ridge regression** (L2 regularization) prevents this by penalizing weight magnitude:

$$\mathcal{L}_{\text{ridge}}(\mathbf{w}) = \frac{1}{n}\|\mathbf{y} - \mathbf{X}\mathbf{w}\|^2 + \lambda \|\mathbf{w}_{1:}\|^2$$

Setting the gradient to zero yields the modified normal equation:

$$\hat{\mathbf{w}}_{\text{ridge}} = (\mathbf{X}^\top\mathbf{X} + \lambda \mathbf{I})^{-1} \mathbf{X}^\top\mathbf{y}$$

The intercept $w_0$ is **not** penalized. Larger $\lambda$ shrinks all feature coefficients toward zero (higher bias, lower variance). Full treatment of regularization appears in **02-02** and **Module 4**.

In [ ]:
def fit_ridge_normal(
    X_design: np.ndarray,
    y: np.ndarray,
    alpha: float,
) -> np.ndarray:
    """Fit ridge regression via the modified normal equation.

    Solves: w* = (X^T X + alpha * I)^{-1} X^T y
    The intercept (index 0) is excluded from regularization.

    Args:
        X_design: Design matrix of shape (n_samples, n_params) with bias column.
        y: Target vector of shape (n_samples,).
        alpha: L2 regularization strength (lambda >= 0).

    Returns:
        Weight vector of shape (n_params,).
    """
    n_params: int = X_design.shape[1]
    # Regularization identity matrix — zero out intercept entry
    reg_matrix: np.ndarray = alpha * np.eye(n_params)
    reg_matrix[0, 0] = 0.0  # do not penalize intercept
    return np.linalg.solve(X_design.T @ X_design + reg_matrix, X_design.T @ y)


# Build design matrices for the full 8-feature dataset
X_train_design: np.ndarray = np.column_stack(
    [np.ones(X_train_scaled.shape[0]), X_train_scaled]
)
X_test_design: np.ndarray = np.column_stack(
    [np.ones(X_test_scaled.shape[0]), X_test_scaled]
)

alphas_ridge: list[float] = [0.0, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
ridge_rows: list[dict] = []

for alpha_val in alphas_ridge:
    w_ridge: np.ndarray = fit_ridge_normal(X_train_design, y_train, alpha_val)
    y_pred_ridge: np.ndarray = X_test_design @ w_ridge
    test_mse_r = float(mean_squared_error(y_test, y_pred_ridge))
    weight_norm = float(np.linalg.norm(w_ridge[1:]))
    ridge_rows.append({
        "lambda (alpha)": alpha_val,
        "Test MSE": round(test_mse_r, 4),
        "||w|| (weight norm)": round(weight_norm, 4),
    })

# Also compare with sklearn Ridge for validation
sklearn_ridge = Ridge(alpha=1.0, fit_intercept=True)
sklearn_ridge.fit(X_train_scaled, y_train)
sk_ridge_mse = float(mean_squared_error(y_test, sklearn_ridge.predict(X_test_scaled)))

ridge_df = pd.DataFrame(ridge_rows)
print("Ridge Regularization Preview — Effect of lambda on Test MSE and Weight Norm:")
print(ridge_df.to_string(index=False))
print(f"\nsklearn Ridge(alpha=1.0) test MSE: {sk_ridge_mse:.4f}")
print(f"From-scratch Ridge(alpha=1.0) test MSE: {ridge_rows[6]['Test MSE']:.4f}")
print("\nObservation: very small lambda -> same as OLS. Very large lambda -> over-shrinks.")

---
## Part 4 — Evaluation & Analysis

We now rigorously evaluate model performance and diagnose residuals to understand where the linear assumption breaks down. The residual analysis is as important as the headline metrics — it reveals *how* the model fails, not just *how much*.

In [ ]:
def compute_regression_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    model_name: str,
) -> dict:
    """Compute standard regression evaluation metrics.

    Args:
        y_true: Ground truth target values of shape (n_samples,).
        y_pred: Predicted values of shape (n_samples,).
        model_name: Identifier label for this row in the summary table.

    Returns:
        Dictionary with model name, MSE, RMSE, MAE, and R^2.
    """
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return {"Model": model_name, "MSE": round(mse, 4), "RMSE": round(rmse, 4),
            "MAE": round(mae, 4), "R2": round(r2, 4)}


# Baseline: always predict the training-set mean
y_mean_pred: np.ndarray = np.full_like(y_test, fill_value=y_train.mean())

metrics_rows = [
    compute_regression_metrics(y_test, y_pred_normal_test,  "From Scratch (Normal Eq)"),
    compute_regression_metrics(y_test, y_pred_gd_test,      "From Scratch (Batch GD)"),
    compute_regression_metrics(y_test, y_pred_sgd_test,     "From Scratch (Mini-Batch SGD)"),
    compute_regression_metrics(y_test, y_pred_sklearn,      "sklearn LinearRegression"),
    compute_regression_metrics(y_test, y_mean_pred,         "Baseline (Predict Mean)"),
]

metrics_df = pd.DataFrame(metrics_rows)
print("Test Set Performance Summary:")
print(metrics_df.to_string(index=False))
print()
print("RMSE is in units of the target ($100k house value).")
print("R2 = 1.0 -> perfect; R2 = 0.0 -> no better than predicting the mean.")
print(f"Our model explains ~{metrics_rows[0]['R2']*100:.0f}% of the variance in house prices.")

In [ ]:
# ── Core Residual Diagnostics ─────────────────────────────────────────────────
residuals: np.ndarray = y_test - y_pred_normal_test

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Predicted vs Actual — points should cluster along the diagonal
val_min = min(y_test.min(), y_pred_normal_test.min())
val_max = max(y_test.max(), y_pred_normal_test.max())
axes[0].scatter(y_pred_normal_test, y_test, alpha=0.06, s=5, color="steelblue")
axes[0].plot([val_min, val_max], [val_min, val_max], "r--", linewidth=1.5,
             label="Perfect fit (y = y_hat)")
axes[0].set_xlabel("Predicted (y_hat)")
axes[0].set_ylabel("Actual (y)")
axes[0].set_title("Predicted vs Actual")
axes[0].legend()

# 2. Residuals vs Fitted — flat zero line means no systematic bias
axes[1].scatter(y_pred_normal_test, residuals, alpha=0.06, s=5, color="steelblue")
axes[1].axhline(0.0, color="red", linewidth=1.5, linestyle="--")
axes[1].set_xlabel("Fitted Values (y_hat)")
axes[1].set_ylabel("Residual (y - y_hat)")
axes[1].set_title("Residuals vs Fitted Values")

# 3. Residual histogram — should be roughly bell-shaped if Gaussian noise holds
axes[2].hist(residuals, bins=70, color="steelblue", edgecolor="white", density=True)
axes[2].axvline(0.0, color="red", linestyle="--", linewidth=1.5)
x_norm = np.linspace(residuals.min(), residuals.max(), 200)
norm_pdf = (1.0 / (residuals.std() * np.sqrt(2 * np.pi))) * np.exp(
    -0.5 * ((x_norm - residuals.mean()) / residuals.std()) ** 2
)
axes[2].plot(x_norm, norm_pdf, "r-", linewidth=2.0, label="Gaussian fit")
axes[2].set_xlabel("Residual")
axes[2].set_ylabel("Density")
axes[2].set_title("Residual Distribution")
axes[2].legend()

plt.suptitle("Residual Diagnostics — Normal Equation on California Housing", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Residual mean   : {residuals.mean():.4f}  (should be near 0)")
print(f"Residual std    : {residuals.std():.4f}")
print(f"Skewness        : {pd.Series(residuals).skew():.4f}  (0 = symmetric)")
print(f"Kurtosis        : {pd.Series(residuals).kurt():.4f}  (0 = Gaussian-like tails)")
print()
print("Key diagnostics:")
print("  Residuals vs Fitted: fan-shaped spread -> heteroscedasticity (variance grows with price).")
print("  Histogram: right-skewed -> model underestimates expensive properties.")

In [ ]:
# ── Standardized Residuals & Q-Q Plot ────────────────────────────────────────
# Standardized residuals = residual / std(residuals); should be ~N(0,1) under OLS assumptions.
# A Q-Q plot compares the empirical quantiles of the residuals to a reference normal distribution.

std_residuals: np.ndarray = (residuals - residuals.mean()) / residuals.std()
sorted_std_resid: np.ndarray = np.sort(std_residuals)

# Reference: draw a same-size sample from N(0,1) and sort it
np.random.seed(SEED)
reference_normal: np.ndarray = np.sort(np.random.randn(len(sorted_std_resid)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: standardized residuals vs index (check for outlier clusters)
axes[0].scatter(range(len(std_residuals)), std_residuals, alpha=0.1, s=4, color="steelblue")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].axhline(2.0, color="red", linestyle="--", linewidth=1.2, label="|z| = 2")
axes[0].axhline(-2.0, color="red", linestyle="--", linewidth=1.2)
axes[0].axhline(3.0, color="darkred", linestyle=":", linewidth=1.2, label="|z| = 3")
axes[0].axhline(-3.0, color="darkred", linestyle=":", linewidth=1.2)
n_outliers_2 = int(np.sum(np.abs(std_residuals) > 2))
n_outliers_3 = int(np.sum(np.abs(std_residuals) > 3))
axes[0].set_xlabel("Test Sample Index")
axes[0].set_ylabel("Standardized Residual")
axes[0].set_title(f"Standardized Residuals  (|z|>2: {n_outliers_2}, |z|>3: {n_outliers_3})")
axes[0].legend(fontsize=9)

# Right: Q-Q plot against reference N(0,1)
axes[1].scatter(reference_normal, sorted_std_resid, alpha=0.15, s=6, color="steelblue",
                label="Sample quantiles")
diag_range = np.array([reference_normal.min(), reference_normal.max()])
axes[1].plot(diag_range, diag_range, "r-", linewidth=2.0, label="y = x (perfect normal)")
axes[1].set_xlabel("Theoretical N(0,1) Quantiles")
axes[1].set_ylabel("Standardized Residual Quantiles")
axes[1].set_title("Q-Q Plot: Residuals vs Normal Distribution")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Fraction of samples with |z| > 2 : {n_outliers_2/len(std_residuals):.2%}  (expected ~5% for normal)")
print(f"Fraction of samples with |z| > 3 : {n_outliers_3/len(std_residuals):.2%}  (expected ~0.3% for normal)")
print()
print("Q-Q interpretation: S-shape curve deviating from y=x in the tails indicates")
print("heavier tails than Gaussian — consistent with the right-skewed histogram above.")

In [ ]:
# ── Error Analysis: Where Does the Model Fail Most? ──────────────────────────
# Identify the test samples with the largest absolute residuals and inspect their features.

test_df_analysis = pd.DataFrame(X_test, columns=list(housing.feature_names))
test_df_analysis["ActualValue"] = y_test
test_df_analysis["PredictedValue"] = y_pred_normal_test
test_df_analysis["Residual"] = residuals
test_df_analysis["AbsResidual"] = np.abs(residuals)

# Largest over-predictions (model predicts too high)
over_pred = test_df_analysis.nlargest(5, "Residual")[["MedInc", "HouseAge", "Latitude",
                                                        "Longitude", "ActualValue",
                                                        "PredictedValue", "Residual"]]
# Largest under-predictions (model predicts too low)
under_pred = test_df_analysis.nsmallest(5, "Residual")[["MedInc", "HouseAge", "Latitude",
                                                          "Longitude", "ActualValue",
                                                          "PredictedValue", "Residual"]]

print("Worst OVER-predictions (model says higher than actual):")
print(over_pred.round(3).to_string(index=False))
print()
print("Worst UNDER-predictions (model says lower than actual):")
print(under_pred.round(3).to_string(index=False))

# Residual magnitude by actual price quintile
n_bins: int = 5
quantile_bins = pd.qcut(test_df_analysis["ActualValue"], q=n_bins, labels=[
    "Q1 (cheap)", "Q2", "Q3", "Q4", "Q5 (expensive)"
])
residual_by_quintile = test_df_analysis.groupby(quantile_bins, observed=True)["AbsResidual"].mean()
print("\nMean absolute residual by actual price quintile:")
for q, val in residual_by_quintile.items():
    print(f"  {q:<20} : {val:.4f} ($100k avg error)")
print("\nConclusion: The model makes larger errors on expensive homes (non-linear relationship).")

In [ ]:
# ── Geographic Error Map ──────────────────────────────────────────────────────
# California Housing records Latitude and Longitude for every census block group.
# Mapping the residuals geographically reveals spatial error structure —
# a pattern the linear model cannot capture without explicit spatial features.

lat_idx: int = list(housing.feature_names).index("Latitude")
lon_idx: int = list(housing.feature_names).index("Longitude")
test_lat: np.ndarray = X_test[:, lat_idx]
test_lon: np.ndarray = X_test[:, lon_idx]
abs_residuals: np.ndarray = np.abs(residuals)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: actual house values — shows coastal high-price bands
sc0 = axes[0].scatter(test_lon, test_lat, c=y_test, cmap="viridis",
                      s=4, alpha=0.5, vmin=0.5, vmax=5.0)
plt.colorbar(sc0, ax=axes[0], label="Actual Value ($100k)")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].set_title("Actual House Values")

# Middle: model's predicted values — shows smoothed spatial variation
sc1 = axes[1].scatter(test_lon, test_lat, c=y_pred_normal_test, cmap="viridis",
                      s=4, alpha=0.5, vmin=0.5, vmax=5.0)
plt.colorbar(sc1, ax=axes[1], label="Predicted Value ($100k)")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
axes[1].set_title("Predicted House Values (Normal Eq)")

# Right: absolute residuals — bright areas = where the model fails most
p95_resid: float = float(np.percentile(abs_residuals, 95))
sc2 = axes[2].scatter(test_lon, test_lat, c=abs_residuals, cmap="hot_r",
                      s=4, alpha=0.6, vmax=p95_resid)
plt.colorbar(sc2, ax=axes[2], label="|Residual| ($100k)")
axes[2].set_xlabel("Longitude")
axes[2].set_ylabel("Latitude")
axes[2].set_title("Absolute Residual Map")

plt.suptitle("Geographic Error Analysis: California Housing", fontsize=13)
plt.tight_layout()
plt.show()

# Coastal vs inland error quantification
coastal_mask: np.ndarray = test_lon < -119.5   # approximate coastal strip
coastal_mae: float = float(np.mean(abs_residuals[coastal_mask]))
inland_mae: float = float(np.mean(abs_residuals[~coastal_mask]))
print("Geographic error breakdown — coastal vs inland:")
print(f"  Coastal areas (lon < -119.5) MAE : {coastal_mae:.4f} ($100k)")
print(f"  Inland areas  (lon >= -119.5) MAE: {inland_mae:.4f} ($100k)")
print(f"  Coastal/inland ratio             : {coastal_mae / inland_mae:.2f}")

# Latitude-band analysis: north-south error gradient
lat_band_labels: list[str] = ["Far South (LA)", "South-Central", "Central Valley",
                               "North-Central", "North (Bay Area)"]
lat_bins = pd.qcut(pd.Series(test_lat), q=5, labels=lat_band_labels)
lat_error_df = pd.DataFrame({
    "Region": lat_band_labels,
    "Mean |Residual|": [
        round(float(np.mean(abs_residuals[lat_bins.values == label])), 4)
        for label in lat_band_labels
    ],
    "N samples": [int(np.sum(lat_bins.values == label)) for label in lat_band_labels],
})
print("\nMean absolute error by latitude band (south to north):")
print(lat_error_df.to_string(index=False))
print()
print("Geographic patterns the linear model cannot fully capture:")
print("  1. Coastal premium — ocean proximity creates a non-linear price jump.")
print("  2. Bay Area / LA Basin clusters — dense urban spatial effects.")
print("  3. Spatial autocorrelation — neighboring districts have correlated prices.")
print("  These patterns motivate spatial regression or geographic feature engineering.")

In [ ]:
# ── Feature Coefficients & Ablation: Effect of Learning Rate ─────────────────
# On standardized features, coefficient magnitude ~ feature importance.
coef_df = pd.DataFrame({
    "Feature": list(housing.feature_names),
    "Coefficient": model_normal_full.coef_,
    "|Coefficient|": np.abs(model_normal_full.coef_),
}).sort_values("|Coefficient|", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: coefficient bar chart
bar_colors = ["#2ecc71" if c > 0 else "#e74c3c" for c in coef_df["Coefficient"]]
axes[0].barh(coef_df["Feature"], coef_df["Coefficient"], color=bar_colors)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Coefficient (standardized features -> comparable scale)")
axes[0].set_title("Linear Regression Coefficients")

# Right: ablation — learning rate effect on batch GD convergence
lr_vals: list[float] = [0.001, 0.01, 0.05, 0.1, 0.3]
for lr_v in lr_vals:
    m = LinearRegressionScratch(
        method="gradient_descent", learning_rate=lr_v, num_epochs=300, tolerance=1e-9
    )
    m.fit(X_train_scaled, y_train)
    final_mse = float(mean_squared_error(y_test, m.predict(X_test_scaled)))
    axes[1].plot(m.loss_history_,
                 label=f"lr={lr_v}  (test={final_mse:.3f})")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Train MSE")
axes[1].set_title("Ablation: Learning Rate vs Convergence Speed (300 epochs)")
axes[1].set_yscale("log")
axes[1].legend(fontsize=8)
axes[1].set_xlim(0, 300)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Feature coefficients (sorted by |magnitude|):")
print(coef_df[["Feature", "Coefficient"]].round(4).to_string(index=False))
print(f"\nIntercept: {model_normal_full.intercept_:.4f}  (~mean house value after standardization)")
print("\nMedInc dominates — income is the strongest predictor of house price.")

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **OLS is MLE under Gaussian noise.** Minimizing squared residuals is not an arbitrary choice — it follows directly from assuming $\epsilon_i \sim \mathcal{N}(0, \sigma^2)$. This probabilistic framing recurs throughout: logistic regression (2-02) replaces Gaussian with Bernoulli, yielding cross-entropy loss.

2. **Two paths to the same solution.** The normal equation $(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$ is exact but costs $\mathcal{O}(p^3)$. Gradient descent (full-batch or mini-batch) iterates to the same minimum and scales to any dataset size — at the cost of a learning rate hyperparameter and many more steps.

3. **Standardize features before gradient descent.** Un-scaled features produce an elongated loss landscape where GD oscillates. After `StandardScaler`, the landscape is near-spherical and a learning rate of $\alpha=0.1$ converges in $\sim$200 epochs.

4. **Polynomial features reveal the bias-variance tradeoff.** Adding $x^2, x^3, \ldots$ reduces training error monotonically but test error has a U-shape — the minimum is the sweet spot between underfitting and overfitting. This is the defining challenge of all of machine learning.

5. **Residual analysis diagnoses model failures.** The fan-shaped residual plot reveals heteroscedasticity — the linear model's error grows with predicted price, indicating a non-linear true relationship. R² = 0.60 means ~40% of house-price variance is still unexplained by a linear model.

---

### What's Next

-> **02-02 (Logistic Regression & Binary Classification)** applies the same MLE framework to a binary outcome, replacing the Gaussian with a Bernoulli likelihood, deriving the sigmoid activation and binary cross-entropy loss, and introducing L1 regularization (Lasso) alongside L2 (Ridge).

---

### Going Further

- **Gauss-Markov theorem:** OLS is the Best Linear Unbiased Estimator (BLUE) among all linear estimators — formalized in **02-06 (Generalized Linear Models)**.
- **Weighted least squares (WLS):** When residual variance is non-constant (as seen in our residual plots), WLS assigns higher weight to low-noise observations, improving efficiency.
- **Stochastic & mini-batch SGD at scale:** The mini-batch SGD introduced here becomes the backbone of all neural network training in **Module 5** — the only differences are the loss function, model architecture, and use of autograd.